___
# <center>Exercícios: medidas de erro e regressão logística</center>
___

## Aula 22

**Objetivo:** ao final destes exercícios, você deve ser capaz de:

 * calcular acurácia, sensibilidade e precisão a partir de uma matriz de
   confusão, e interpretar cada uma no caso;
 * escolher a medida que mais importa para um problema, e o ponto de corte;
 * interpretar a saída de uma regressão logística, inclusive com variável
   categórica, e calcular uma probabilidade prevista a partir dela;
 * explicar por que um coeficiente muda quando entra outra variável;
 * desconfiar de um modelo que acerta muito nos dados em que foi ajustado.

**Vale até 4 pontos extras no Projeto 06**, 1 por exercício. É individual:
pode conversar com colegas, mas escreva com as suas palavras.

**Você não precisa escrever código de modelo.** As células de código já estão
prontas: é só rodar. Onde houver conta, use o Python como calculadora (por
exemplo, `124 / 179`). As respostas escritas vão nas células cinzas que dizem
"Escreva aqui a sua resposta".

**Nome:**

**E-mail Insper:**

**Entrega:** ao terminar, baixe este notebook (*Arquivo > Fazer download >
Baixar o .ipynb*) e envie no BlackBoard, no item **Exercícios da aula 22**.

___
<div id="indice"></div>

## Índice

- [As bases](#bases)
- [Teoria: o que você precisa para os exercícios](#teoria)
- [Exercício 1: reclamações trabalhistas, quantas terminam em acordo?](#ex1)
- [Exercício 2: pedidos de benefício, onde está a fraude?](#ex2)
- [Exercício 3: ações contra companhias aéreas, o que explica a procedência?](#ex3)
- [Exercício 4: juizado especial, um modelo que decorou o histórico](#ex4)
- [Respostas numéricas esperadas](#respostas)

___
<div id="bases"></div>

## As bases

**Todas as bases são simuladas.** Os números ilustram os métodos e não dizem
nada sobre processos reais. Rode a célula abaixo antes de começar.

In [ ]:
# @title Rode esta célula para criar as bases (não precisa ler o código)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

rng = np.random.default_rng(2026)
sig = lambda z: 1 / (1 + np.exp(-z))

# exercício 1: reclamações trabalhistas
n = 400
anos = rng.integers(1, 15, n)
valor = rng.gamma(4, 10, n).round(1)
testemunha = rng.binomial(1, .5, n)
acordo = rng.binomial(1, sig(1.7 - 0.06 * valor + 0.15 * anos - 1.0 * testemunha))
trabalhista = pd.DataFrame({"acordo": acordo, "valor_pedido": valor,
                            "anos_empresa": anos, "testemunha": testemunha})

# exercício 2: pedidos de benefício
n = 2000
dias = rng.integers(0, 60, n)
divergencia = rng.binomial(1, .3, n)
procurador = rng.binomial(1, .1, n)
fraude = rng.binomial(1, sig(-5.0 + 0.04 * dias + 1.6 * divergencia + 1.8 * procurador))
beneficios = pd.DataFrame({"fraude": fraude, "dias_cadastro": dias,
                           "divergencia": divergencia, "procurador_frequente": procurador})

# exercício 3: ações contra companhias aéreas
n = 600
problema = rng.choice(["atraso", "cancelamento", "bagagem"], n, p=[.5, .3, .2])
advogado = rng.binomial(1, .5, n)
comprovante = rng.binomial(1, np.where(advogado == 1, .8, .3))
z = (-0.9 + np.select([problema == "cancelamento", problema == "bagagem"], [0.7, 1.2], 0)
     + 0.2 * advogado + 1.5 * comprovante)
aereas = pd.DataFrame({"procedente": rng.binomial(1, sig(z)), "problema": problema,
                       "advogado": advogado, "comprovante": comprovante})

# exercício 4: juizado especial cível
n = 320
juiz = rng.integers(1, 81, n)
efeito_juiz = rng.normal(0, 0.3, 81)
prova = rng.binomial(1, .5, n)
valor = rng.gamma(3, 3, n).round(1)
procedente = rng.binomial(1, sig(-0.8 + 2.0 * prova - 0.04 * valor + efeito_juiz[juiz]))
juizado = pd.DataFrame({"procedente": procedente, "prova_documental": prova,
                        "valor_pedido": valor, "juiz": [f"J{j:02d}" for j in juiz]})
ordem = rng.permutation(n)
historico, novos = juizado.iloc[ordem[:220]], juizado.iloc[ordem[220:]]


def razoes_de_chance(modelo):
    """Coeficiente, razão de chances e valor-p de cada variável."""
    return pd.DataFrame({"coeficiente": modelo.params, "razao_de_chances": np.exp(modelo.params),
                         "valor_p": modelo.pvalues}).round(3)


def medidas_por_corte(y, prob, cortes):
    """Matriz de confusão e medidas para cada corte."""
    linhas = []
    for c in cortes:
        sinal = prob >= c
        vp, fp = int((sinal & (y == 1)).sum()), int((sinal & (y == 0)).sum())
        fn, vn = int((~sinal & (y == 1)).sum()), int((~sinal & (y == 0)).sum())
        linhas.append({"corte": c, "sinalizados": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
                       "acuracia": (vp + vn) / len(y), "precisao": vp / max(vp + fp, 1),
                       "sensibilidade": vp / (vp + fn)})
    return pd.DataFrame(linhas).round(3)


def _colunas(dados, com_juiz):
    x = dados[["prova_documental", "valor_pedido"]].copy()
    if com_juiz:
        juizes = sorted(juizado["juiz"].unique())
        x = x.join(pd.get_dummies(dados["juiz"], dtype=float).reindex(columns=juizes, fill_value=0))
    return x


def ajusta_juizado(dados, com_juiz):
    """Regressão logística, com ou sem o juiz como variável."""
    modelo = LogisticRegression(C=1e4, max_iter=20000).fit(_colunas(dados, com_juiz), dados["procedente"])
    modelo.com_juiz = com_juiz
    return modelo


def prob_juizado(modelo, dados):
    return modelo.predict_proba(_colunas(dados, modelo.com_juiz))[:, 1]


def taxa_de_acerto(modelo, dados):
    """Proporção de acertos do modelo na base, com corte 0,5."""
    return round(((prob_juizado(modelo, dados) >= 0.5) == dados["procedente"]).mean(), 3)


print("Bases criadas: trabalhista, beneficios, aereas, juizado (historico e novos).")

[Volta ao Índice](#indice)

___
<div id="teoria"></div>

# Teoria: o que você precisa para os exercícios

### 1. Da probabilidade à decisão

Um modelo de classificação, como a regressão logística, dá para cada caso uma
**probabilidade** do evento (acordo, fraude, procedência). Para virar decisão,
escolhemos um **ponto de corte**: o caso é previsto como "sim" quando a
probabilidade é maior ou igual ao corte. O corte é uma **escolha**, e não sai
do modelo.

### 2. Matriz de confusão

Comparando o previsto com o que aconteceu de fato, cada caso cai numa de quatro
casas:

| | previsto "não" | previsto "sim" |
|---|---|---|
| **real "não"** | VN: verdadeiro negativo | FP: falso positivo |
| **real "sim"** | FN: falso negativo | VP: verdadeiro positivo |

- **Falso positivo:** o modelo disse "sim" e era "não" (um alarme à toa).
- **Falso negativo:** o modelo disse "não" e era "sim" (um caso que escapou).

### 3. Medidas de erro

$$\text{acurácia} = \frac{VP + VN}{\text{total}} \qquad
\text{sensibilidade} = \frac{VP}{VP + FN} \qquad
\text{precisão} = \frac{VP}{VP + FP}$$

- **Acurácia:** de todos os casos, quantos o modelo acertou.
- **Sensibilidade** (ou revocação): dos casos que **eram** "sim", quantos o
  modelo encontrou. O denominador é a linha do "sim" real.
- **Precisão:** dos casos que o modelo **apontou** como "sim", quantos eram
  mesmo. O denominador é a coluna do "sim" previsto.

**Exemplo de interpretação.** Um tribunal usa um modelo para separar recursos
que provavelmente repetem uma tese já julgada. Em 200 recursos: VP = 40,
FP = 10, FN = 20, VN = 130.

- Acurácia = (40 + 130) / 200 = 0,85. *"O modelo classificou corretamente 85%
  dos recursos."*
- Sensibilidade = 40 / (40 + 20) = 0,67. *"Dos recursos que de fato repetiam a
  tese, o modelo separou 2 em cada 3; o outro terço passou sem ser separado."*
- Precisão = 40 / (40 + 10) = 0,80. *"Dos recursos que o modelo separou, 80%
  repetiam mesmo a tese; os outros 20% foram separados à toa."*

Repare que a boa interpretação diz **quem é o denominador** ("dos que eram",
"dos que o modelo apontou") e usa as palavras do caso, não as do modelo.

### 4. O modelo trivial e o evento raro

O **modelo trivial** prevê a mesma resposta para todo mundo, sempre a mais
comum. Ele não usa nenhuma informação, e serve de **régua**: um modelo útil
tem que fazer melhor que ele.

- Se o evento é **comum** (perto de metade dos casos), o trivial acerta perto
  de 50%, e a acurácia é uma boa forma de comparar.
- Se o evento é **raro** (por exemplo, 5% dos casos), o trivial que diz sempre
  "não" acerta 95% sem encontrar **nenhum** caso. A acurácia fica alta e
  **engana**: olhe a sensibilidade e a precisão.

### 5. Qual medida importa, e qual corte usar

Depende de **quanto custa cada erro**, e de quem paga:

- se o caso que escapa (FN) é o erro grave, a **sensibilidade** importa mais:
  baixe o corte para apontar mais casos;
- se o alarme à toa (FP) é caro, ou a equipe que analisa os casos apontados é
  pequena, a **precisão** importa mais: suba o corte.

Baixar o corte aponta mais casos: a sensibilidade sobe e a precisão
normalmente cai. Quando há um limite de **capacidade** (só dá para analisar N
casos), o corte mais baixo possível é o que aponta no máximo N casos.

### 6. Regressão logística

A regressão logística modela a probabilidade $p$ do evento. Ela trabalha com a
**chance** (*odds*), que é a probabilidade de acontecer dividida pela de não
acontecer:

$$\text{chance} = \frac{p}{1 - p} \qquad \text{(por exemplo, } p = 0{,}8 \Rightarrow \text{chance} = 4\text{, ou "4 para 1")}$$

O modelo é uma **combinação linear** das variáveis, como na regressão linear,
mas para o logaritmo da chance (o *logit*):

$$z = b_0 + b_1 x_1 + b_2 x_2 + \dots \qquad\qquad p = \frac{1}{1 + e^{-z}}$$

A segunda fórmula, a **função logística**, transforma qualquer número $z$ numa
probabilidade entre 0 e 1. Ela está pronta na célula abaixo, com alguns
exemplos e o gráfico: $z = 0$ dá $p = 0{,}5$; $z$ positivo dá mais de 50%; $z$
negativo, menos.

In [ ]:
def logistica(z):
    """Transforma a soma z (o logit) numa probabilidade entre 0 e 1."""
    return 1 / (1 + np.exp(-z))

for z in [-3, -1, 0, 1, 3]:
    print(f"z = {z:>2}  ->  p = {logistica(z):.3f}")

z = np.linspace(-6, 6, 200)
plt.plot(z, logistica(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.xlabel("z (soma da combinação linear)")
plt.ylabel("probabilidade p")
plt.title("Função logística")
plt.show()

**Razão de chances.** Cada coeficiente $b$ vira uma **razão de chances**
$e^{b}$: quantas vezes a chance fica multiplicada quando a variável aumenta 1
(ou passa de 0 para 1), **mantidas as demais variáveis fixas**.

- razão de chances **maior que 1**: a chance aumenta;
- **menor que 1**: a chance diminui;
- **igual a 1** (coeficiente zero): não muda. O **valor-p** diz se a diferença
  para 1 é distinguível do acaso (usamos 5%).

**Variável categórica.** Uma variável com categorias (por exemplo, o tipo de
ação: `cobranca`, `despejo`, `indenizacao`) entra no modelo com uma categoria
de **referência**. Cada outra categoria ganha o seu coeficiente, **sempre em
comparação com a referência**. Na saída do `statsmodels`, a linha
`C(tipo, Treatment('cobranca'))[T.despejo]` é o despejo comparado com a
cobrança.

**Exemplo de interpretação.** Num modelo de procedência em ações de consumo, a
razão de chances de `nota_fiscal` (1 se o consumidor juntou nota fiscal) é
2,0, com valor-p 0,001. *"Mantidas as demais variáveis, a chance de
procedência de quem juntou nota fiscal é o dobro da chance de quem não
juntou."* Se a razão de chances de `[T.despejo]` fosse 0,5: *"a chance de
procedência num despejo é metade da chance numa cobrança, mantido o resto."*

### 7. Quando entra outra variável

O coeficiente de uma variável mede o seu efeito **mantidas fixas as outras
variáveis do modelo**. Se uma variável importante fica de fora e está
associada a outra que está dentro, a de dentro "leva o crédito" da de fora.
Quando a variável que faltava entra, o coeficiente da primeira **muda**, às
vezes muito.

### 8. Treino e teste

Um modelo aprende com uma base de dados. Medir o acerto nessa mesma base é
como corrigir uma prova cujas respostas o aluno viu antes: o resultado sai
otimista. O teste honesto é em **casos novos**, que o modelo não viu. O
exercício 4 mostra o que pode dar errado.

[Volta ao Índice](#indice)

___
<div id="ex1"></div>

## Exercício 1: reclamações trabalhistas, quantas terminam em acordo?

Uma vara do trabalho quer prever quais reclamações vão terminar em **acordo**,
para marcar audiência de conciliação logo no início só nesses processos. Na
base `trabalhista`, cada linha é uma reclamação:

| coluna | o que é |
|---|---|
| `acordo` | 1 se terminou em acordo, 0 se não |
| `valor_pedido` | valor pedido na reclamação, em R\$ mil |
| `anos_empresa` | anos de trabalho na empresa |
| `testemunha` | 1 se o reclamante arrolou testemunha |

A regressão logística já está ajustada. O processo é previsto como acordo
quando a probabilidade é de pelo menos 0,5. Aqui o "sim" é o acordo.

In [ ]:
modelo1 = smf.logit("acordo ~ valor_pedido + anos_empresa + testemunha", data=trabalhista).fit(disp=0)
trabalhista["prob"] = modelo1.predict(trabalhista)
trabalhista["previsto"] = (trabalhista["prob"] >= 0.5).astype(int)

trabalhista["acordo"].mean()          # proporção de acordos na base

In [ ]:
# MATRIZ DE CONFUSÃO
pd.crosstab(trabalhista["acordo"], trabalhista["previsto"], margins=True)   # linhas: real; colunas: previsto

<div id="ex1a"></div>

**(a)** Usando os números da matriz de confusão, calcule a **acurácia**, a
**sensibilidade** e a **precisão** do modelo. Primeiro identifique VN, FP, FN
e VP na tabela (a linha é o real, a coluna é o previsto).

In [ ]:
# faça aqui as suas contas

<div id="ex1b"></div>

**(b)** **Interprete** a sensibilidade e a precisão que você calculou, no
contexto do caso: o que cada número diz sobre as reclamações e as audiências de
conciliação? Siga o exemplo de interpretação da seção
[Teoria](#teoria) (item 3): diga quem é o denominador e use as palavras do caso.

<div id="ex1c"></div>

**(c)** Nessa base, perto de metade das reclamações (45%) termina em acordo.
Qual seria a acurácia do **modelo trivial**, que prevê sempre a resposta mais
comum? A acurácia é uma boa medida para comparar o modelo com ele neste caso?
Por quê?

In [ ]:
# faça aqui as suas contas

[Volta ao Índice](#indice)

___
<div id="ex2"></div>

## Exercício 2: pedidos de benefício, onde está a fraude?

Uma equipe de auditoria previdenciária recebe **2.000 pedidos de benefício**
por mês e quer separar os suspeitos de **fraude** para análise manual. Aqui o
"sim" é a fraude. Na base `beneficios`:

| coluna | o que é |
|---|---|
| `fraude` | 1 se o pedido era fraudulento, 0 se não |
| `dias_cadastro` | dias entre o cadastro do segurado e o pedido |
| `divergencia` | 1 se houve divergência nos documentos |
| `procurador_frequente` | 1 se o pedido foi feito por procurador com muitos pedidos |

A função `medidas_por_corte` já calcula a matriz de confusão (VP, FP, FN, VN)
e as medidas para cada corte que você pedir.

In [ ]:
modelo2 = smf.logit("fraude ~ dias_cadastro + divergencia + procurador_frequente", data=beneficios).fit(disp=0)
beneficios["prob"] = modelo2.predict(beneficios)

beneficios["fraude"].mean()          # proporção de fraudes na base

In [ ]:
medidas_por_corte(beneficios["fraude"], beneficios["prob"], [0.5])

<div id="ex2a"></div>

**(a)** Qual seria a acurácia do **modelo trivial**, que diz que **nenhum**
pedido é fraude? Compare com a acurácia do modelo no corte 0,5 e com a
sensibilidade dele nesse corte. O modelo com corte 0,5 é útil para a equipe?

In [ ]:
# faça aqui as suas contas

<div id="ex2b"></div>

**(b)** A equipe consegue analisar até **350 pedidos por mês**. Rode a tabela
abaixo e escolha o corte mais baixo que cabe nessa capacidade (a coluna
`sinalizados` é quantos pedidos vão para a equipe). Nesse corte, quantas
fraudes a equipe encontra e quantas deixa passar? De cada 10 pedidos
analisados, quantos são fraude de verdade?

In [ ]:
medidas_por_corte(beneficios["fraude"], beneficios["prob"], [0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50])

<div id="ex2c"></div>

**(c)** Neste caso, os dois erros são:

- **falso positivo:** um pedido **honesto** sinalizado como suspeito. Vai para
  análise manual à toa: gasta o tempo da equipe e atrasa o benefício de quem
  tem direito;
- **falso negativo:** uma **fraude** que não foi sinalizada. O benefício é
  pago indevidamente, às vezes por anos.

Responda: (i) qual medida mais importa para a equipe, a **acurácia**, a
**sensibilidade** ou a **precisão**? Justifique com o custo de cada erro.
(ii) **Novo cenário:** a equipe dobra de tamanho e passa a analisar até **700
pedidos por mês**. Pela tabela do item (b), que corte você escolheria agora? O
que acontece com a sensibilidade e com a precisão, em comparação com o corte do
item (b)?

[Volta ao Índice](#indice)

___
<div id="ex3"></div>

## Exercício 3: ações contra companhias aéreas, o que explica a procedência?

Um juizado recebe muitas ações de passageiros contra companhias aéreas, pedindo
indenização. Queremos saber o que está associado à **procedência** do pedido.
Na base `aereas`:

| coluna | o que é |
|---|---|
| `procedente` | 1 se o pedido foi julgado procedente, 0 se não |
| `problema` | `atraso`, `cancelamento` ou `bagagem` (extravio de bagagem) |
| `advogado` | 1 se o passageiro tinha advogado |
| `comprovante` | 1 se o passageiro juntou comprovantes dos gastos (hotel, alimentação, compras) |

`problema` é **categórica**, com `atraso` como **referência** (veja o item 6 da
[Teoria](#teoria)).

O **modelo 1** usa o tipo de problema e o advogado:

In [ ]:
modelo3a = smf.logit("procedente ~ C(problema, Treatment('atraso')) + advogado", data=aereas).fit(disp=0)
razoes_de_chance(modelo3a)

<div id="ex3a"></div>

**(a)** Na tabela do modelo 1, a linha `...[T.cancelamento]` compara
**cancelamento com atraso**, e a linha `...[T.bagagem]` compara **extravio de
bagagem com atraso**. Interprete a **razão de chances** dessas duas linhas, como
no exemplo da Teoria ("a chance de procedência num ... é ... vezes a chance
num ..."). Qual tipo de problema tem mais chance de procedência?

O **modelo 2** acrescenta a variável `comprovante`. A tabela cruzada logo
abaixo mostra, para quem tem e para quem não tem advogado, a proporção que
juntou comprovante.

In [ ]:
modelo3b = smf.logit("procedente ~ C(problema, Treatment('atraso')) + advogado + comprovante", data=aereas).fit(disp=0)
razoes_de_chance(modelo3b)

In [ ]:
pd.crosstab(aereas["advogado"], aereas["comprovante"], normalize="index")   # linhas: advogado; colunas: comprovante

<div id="ex3b"></div>

**(b)** Responda em três partes:

1. Qual a razão de chances de `advogado` no modelo 1, e qual no modelo 2?
2. Pela tabela cruzada: quem tem advogado junta comprovante com mais ou com
   menos frequência do que quem não tem? Quanto, em cada grupo?
3. Por que a razão de chances do advogado diminuiu quando o comprovante entrou
   no modelo (veja o item 7 da Teoria)? Pelo modelo 2, dá para dizer que ter
   advogado, por si só, aumenta a chance de ganhar?

<div id="ex3c"></div>

**(c)** Vamos calcular, **passo a passo**, a probabilidade de procedência que o
**modelo 2** prevê para um passageiro com **voo cancelado**, **sem advogado** e
**com comprovante**.

**Passo 1: a combinação linear.** Para esse passageiro, as variáveis valem:
cancelamento = 1, bagagem = 0, advogado = 0, comprovante = 1. Então

$$z = b_0 + b_{\text{cancelamento}} \cdot 1 + b_{\text{bagagem}} \cdot 0 + b_{\text{advogado}} \cdot 0 + b_{\text{comprovante}} \cdot 1$$

Copie os **coeficientes** (não as razões de chances) da tabela do modelo 2 e
calcule $z$ na célula abaixo.

In [ ]:
# faça aqui as suas contas

**Passo 2: a probabilidade.** Aplique a função `logistica` (da
[Teoria](#teoria), item 6) ao $z$ do passo 1. Escreva na célula de resposta a
probabilidade e o que ela significa. Se quiser conferir, a última linha da
célula de conta faz a mesma previsão com o próprio modelo.

In [ ]:
# faça aqui a sua conta com logistica(z)


# para conferir:
modelo3b.predict(pd.DataFrame({"problema": ["cancelamento"], "advogado": [0], "comprovante": [1]}))

[Volta ao Índice](#indice)

___
<div id="ex4"></div>

## Exercício 4: juizado especial, um modelo que decorou o histórico

Um escritório quer um modelo para prever se as suas ações de indenização no
juizado especial cível serão **procedentes**. Ele tem um **histórico** de 220
processos já julgados, por **80 juízes** diferentes:

| coluna | o que é |
|---|---|
| `procedente` | 1 se o pedido foi julgado procedente |
| `prova_documental` | 1 se a inicial veio com prova documental |
| `valor_pedido` | valor pedido, em R\$ mil |
| `juiz` | quem julgou (`J01` a `J80`) |

Um advogado do escritório sugere: *"cada juiz tem o seu jeito; vamos colocar o
juiz no modelo"*. O juiz é uma variável categórica com 80 categorias, então o
modelo ganha um coeficiente para cada juiz. Com 220 processos, cada juiz tem,
em média, menos de 3 processos no histórico.

In [ ]:
historico["juiz"].value_counts().describe()      # processos por juiz no histórico

<div id="ex4a"></div>

**(a)** A célula abaixo ajusta a regressão logística no histórico, com a prova,
o valor e o juiz, e mede a **acurácia** do modelo (com corte 0,5) **nos mesmos
220 processos**. Quanto o modelo acerta? Se você fosse o escritório, ficaria
satisfeito com esse modelo?

In [ ]:
modelo_juiz = ajusta_juizado(historico, com_juiz=True)
taxa_de_acerto(modelo_juiz, historico)

**Atenção.** O modelo foi avaliado nos **mesmos processos que usou para
aprender**. É como corrigir uma prova cujas respostas o aluno viu antes: a
nota sai alta, mas não diz se ele aprendeu. O que interessa ao escritório é o
acerto em processos **que o modelo ainda não viu**.

Passaram alguns meses, e chegaram **100 processos novos**, já julgados, na base
`novos`. Eles não foram usados para ajustar o modelo. Agora dá para fazer o
teste honesto.

In [ ]:
taxa_de_acerto(modelo_juiz, novos)

Para entender o que aconteceu, veja o que o modelo aprendeu sobre alguns
juízes. A tabela mostra quantos processos cada juiz tem no histórico, quantos
foram procedentes, e a probabilidade que o modelo dá a um processo **novo**
desse juiz, com prova documental e valor de R\$ 10 mil.

In [ ]:
por_juiz = historico.groupby("juiz")["procedente"].agg(processos="count", procedentes="sum")
caso = pd.DataFrame({"prova_documental": 1, "valor_pedido": 10.0, "juiz": por_juiz.index})
por_juiz["prob_processo_novo"] = prob_juizado(modelo_juiz, caso)
por_juiz.sort_values("prob_processo_novo").iloc[[0, 1, 2, -3, -2, -1]].round(3)

<div id="ex4b"></div>

**(b)** Compare a acurácia do modelo no histórico e nos processos novos. Depois,
olhe a tabela: o que o modelo "aprendeu" sobre os juízes **J76** e **J61**? Você
confiaria nessas probabilidades? Explique, com as suas palavras, por que o
modelo foi tão bem no histórico e pior nos processos novos.

**O nome disso é sobreajuste** (*overfitting*): o modelo se ajusta tão bem aos
dados em que aprendeu que passa a errar mais nos casos novos. Ele aparece
quando o modelo é **complexo demais para a quantidade de dados**: aqui, 80
coeficientes de juiz para 220 processos.

Uma saída é usar um modelo **mais simples**. A célula abaixo ajusta o modelo
**sem o juiz**, só com a prova e o valor, e monta uma tabela com a acurácia dos
dois modelos nas duas bases.

In [ ]:
modelo_simples = ajusta_juizado(historico, com_juiz=False)

pd.DataFrame({
    "modelo": ["com juiz", "sem juiz"],
    "acuracia_historico": [taxa_de_acerto(modelo_juiz, historico), taxa_de_acerto(modelo_simples, historico)],
    "acuracia_novos": [taxa_de_acerto(modelo_juiz, novos), taxa_de_acerto(modelo_simples, novos)],
})

<div id="ex4c"></div>

**(c)** Qual modelo acerta mais no histórico? E nos processos novos? Qual você
recomendaria ao escritório, e por quê? Por fim, dê o seu palpite: se o
escritório quiser mesmo usar a informação do juiz, o que poderia fazer para não
cair no sobreajuste?

**O que este exercício mostra.** Medir o acerto de um modelo na mesma base em
que ele aprendeu dá um resultado **otimista**. Por isso, em machine learning,
separamos uma parte dos dados que o modelo **não vê** durante o ajuste (a base
de **teste**), e escolhemos entre modelos pelo acerto nela. Mais variáveis
nem sempre ajudam: com pouco dado, o modelo complexo decora. Nas próximas
aulas, veremos como fazer essa escolha sem gastar a base de teste (a
**validação cruzada**) e como "encolher" coeficientes (a **regularização**).

[Volta ao Índice](#indice)

___
<div id="respostas"></div>

## Respostas numéricas esperadas

Use para conferir as suas contas. As interpretações não estão aqui.

| item | resposta |
|---|---|
| 1a | acurácia 0,750; sensibilidade 0,693; precisão 0,734 |
| 1c | modelo trivial (sempre "sem acordo"): 0,5525 |
| 2a | trivial: 0,9205; modelo no corte 0,5: acurácia 0,925 e sensibilidade 0,094 (15 de 159 fraudes) |
| 2b | corte 0,15: 324 sinalizados, 76 fraudes encontradas, 83 que passam, precisão 0,235 (cerca de 2 em 10) |
| 2c | com 700 análises, corte 0,08: 555 sinalizados, sensibilidade 0,660, precisão 0,189 |
| 3a | razão de chances: cancelamento 1,81; bagagem 2,26 (ambos contra atraso) |
| 3b | advogado: 3,38 no modelo 1 e 1,50 no modelo 2; comprovante: 78% de quem tem advogado e 28% de quem não tem |
| 3c | z ≈ 1,48; p ≈ 0,81 |
| 4a | acurácia do modelo com juiz no histórico: 0,864 |
| 4b | acurácia do modelo com juiz nos processos novos: 0,69 |
| 4c | modelo sem juiz: 0,777 no histórico e 0,76 nos processos novos |

[Volta ao Índice](#indice)